<a href="https://colab.research.google.com/github/ggalanc/proyecto_dengue/blob/main/notebooks/practico_06_proyecto_final_fase3_fase5_colab_Gerardo_Galan_Mabel_Herrera.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
# Práctico Final: MLOps Local con Monitoreo de Data Drift
## Fase 3 y Fase 5 — Servicio de Inferencia, Dashboard y Plan de Acción (Google Colab)
### Magíster en Ciencia de Datos — Tópicos en Data Science II
---

**Integrantes:** Gerardo Galán - Mabel Herrera

**Continúa de:** Fase 2 (`models/modelo_dengue.joblib`, ya entrenado) y Fase 4 (`reports/*.csv`, métricas de drift ya calculadas).

**Qué hace este notebook:**
1. Levanta el servicio de inferencia (FastAPI) de la Fase 3 dentro de esta misma sesión de Colab.
2. Lo alimenta con la producción simulada, exactamente como en una demo local.
3. Prueba en vivo el gatillo de reentrenamiento y el rollback de la Fase 5.
4. Expone el dashboard de Streamlit con una URL pública (vía `serveo.net`, sin cuenta ni token) para poder verlo desde el navegador.


---
### Nota sobre por qué esto corre distinto en Colab que en una máquina local

Localmente, el servicio, el dashboard y el script que los alimenta corren como **procesos separados** en distintas terminales. Colab es un solo notebook con un solo proceso, así que:

- El servicio FastAPI se levanta en un **hilo en segundo plano** dentro de esta misma sesión, en vez de una terminal aparte.
- El dashboard de Streamlit necesita una **URL pública** (`serveo.net`) para poder verse, porque no hay forma de abrir `localhost` directamente desde Colab.
- Esta sesión de Colab es **efímera**: todo lo que se genera (modelo reentrenado, base de datos, reportes) vive solo mientras dure esta ejecución. Por eso este notebook está pensado para correrse de principio a fin en una sola sesión continua — igual que será la demo en vivo de la defensa oral — y no para dejar resultados guardados entre sesiones distintas.


---
## Setup — Traer el repositorio a Colab e instalar dependencias


In [ ]:
# ── Bootstrap para Google Colab: trae el repositorio a este entorno ──
import os

REPO_URL = "https://github.com/ggalanc/proyecto_dengue.git"
REPO_DIR = "/content/dengue_mlops_drift_repo_1"

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB and not os.path.exists(REPO_DIR):
    if REPO_URL:
        print("Clonando repositorio...")
        get_ipython().system('git clone -q "{}" "{}"'.format(REPO_URL, REPO_DIR))
    else:
        print("REPO_URL esta vacio. Selecciona el .zip del repositorio para subirlo:")
        from google.colab import files
        subido = files.upload()
        zip_name = list(subido.keys())[0]
        os.makedirs(REPO_DIR, exist_ok=True)
        get_ipython().system('unzip -q "{}" -d "{}"'.format(zip_name, REPO_DIR))
        contenido = os.listdir(REPO_DIR)
        if len(contenido) == 1 and os.path.isdir(os.path.join(REPO_DIR, contenido[0])):
            sub = os.path.join(REPO_DIR, contenido[0])
            for item in os.listdir(sub):
                os.rename(os.path.join(sub, item), os.path.join(REPO_DIR, item))
            os.rmdir(sub)

if IN_COLAB:
    os.chdir(REPO_DIR)  # aqui NO entramos a notebooks/ -- necesitamos service/ y dashboard/ como hermanos
    print("Directorio de trabajo:", os.getcwd())
else:
    print("No se detecto Colab -- se asume ejecucion local normal, sin cambios de directorio.")


In [ ]:
# ── Dependencias que Colab NO trae preinstaladas ──
# (pandas, numpy, scipy, scikit-learn, matplotlib, seaborn y joblib ya vienen listos en Colab)
get_ipython().system('pip install -q fastapi "uvicorn[standard]" httpx pydantic streamlit')
print("Dependencias instaladas")


---
## Fase 3 — Levantar el servicio de inferencia dentro de Colab

Usamos el modelo que ya viene entrenado en `models/modelo_dengue.joblib` (Fase 2) — no hace falta reentrenar para esta demo. `service/app.py` es exactamente el mismo archivo que se usaría con `uvicorn app:app --port 8000` en una máquina local; aquí simplemente lo corremos en un hilo en vez de en una terminal aparte.


In [ ]:
# ── Levantar el servicio FastAPI en un hilo de fondo (version robusta) ──
import sys, threading, time
import requests

sys.path.insert(0, os.path.join(REPO_DIR, "service"))
sys.path.insert(0, REPO_DIR)
os.chdir(os.path.join(REPO_DIR, "service"))

from app import app as fastapi_app
import uvicorn

HEALTH_URL = "http://127.0.0.1:8000/health"

def _iniciar_servicio():
    uvicorn.run(fastapi_app, host="127.0.0.1", port=8000, log_level="warning")

def _servicio_responde():
    try:
        r = requests.get(HEALTH_URL, timeout=2)
        return r
    except requests.exceptions.ConnectionError:
        return None

# si esta celda ya se corrio antes en esta misma sesion, reusamos el servicio en vez de duplicarlo
r = _servicio_responde()
if r is not None:
    print("El servicio ya estaba corriendo:", r.status_code)
    print(r.json())
else:
    server_thread = threading.Thread(target=_iniciar_servicio, daemon=True)
    server_thread.start()

    # Colab a veces tarda en arrancar el servicio (mas si se acaban de instalar
    # dependencias en esta misma sesion) -- reintentamos hasta 20 segundos en vez
    # de asumir un tiempo fijo.
    r = None
    for _ in range(20):
        time.sleep(1)
        r = _servicio_responde()
        if r is not None:
            break

    if r is not None:
        print("Estado del servicio:", r.status_code)
        print(r.json())
    else:
        print(
            "El servicio no respondio despues de 20 segundos.\n"
            "Esto suele pasar la PRIMERA vez que se instalan dependencias nuevas en esta sesion.\n"
            "Solucion: Entorno de ejecucion > Reiniciar sesion, y despues Entorno de ejecucion > Ejecutar todas."
        )


---
## Alimentar el servicio con la producción simulada (demo real de Fase 3)

Corremos `service/simular_produccion.py` sin modificarlo — le manda al servicio, semana por semana, las 292 observaciones de `data/processed/produccion_simulada.csv`, exactamente como en la demo local.


In [ ]:
# ── Alimentar el servicio (puede tardar 1-2 minutos: son 292 peticiones reales) ──
if _servicio_responde() is None:
    print(
        "El servicio no responde en http://127.0.0.1:8000 -- vuelvan a ejecutar la celda "
        "anterior ('Levantar el servicio FastAPI...') antes de continuar."
    )
else:
    os.chdir(os.path.join(REPO_DIR, "service"))
    get_ipython().system('python simular_produccion.py --url http://127.0.0.1:8000')


In [ ]:
# ── Confirmar cuántas predicciones quedaron registradas ──
r = requests.get("http://127.0.0.1:8000/health")
print(r.json())


---
## Puente a la Fase 4 (métricas de drift)

El cálculo de PSI/KS por ventana y las alertas (`reports/drift_detalle.csv`, `reports/mae_por_ventana.csv`, `reports/alertas_por_ventana.csv`) ya está hecho y viene incluido en el repositorio — el dashboard de más abajo simplemente los lee, igual que en la versión local.

Si quieren **regenerarlos desde cero** en esta misma sesión de Colab (por ejemplo, porque acaban de reentrenar el modelo más abajo y quieren que el dashboard refleje el modelo nuevo), corran esta celda — ejecuta el notebook de Fase 4 completo de forma no interactiva, igual que sugiere el README para la opción de reentrenar desde cero:


In [ ]:
# ── OPCIONAL: regenerar reports/ ejecutando el notebook de Fase 4 completo ──
REGENERAR_REPORTS = False  # cambien a True si quieren recalcular el drift con el modelo actual

if REGENERAR_REPORTS:
    os.chdir(REPO_DIR)
    nb = "notebooks/practico_06_proyecto_final_fase4_drift_Gerardo_Galan_Mabel_Herrera.ipynb"
    get_ipython().system('jupyter nbconvert --to notebook --execute --inplace "{}"'.format(nb))
    print("reports/ regenerado con el modelo actual")
else:
    print("Se usan los reports/ ya incluidos en el repositorio (sin regenerar)")


---
## Fase 5 — Probar el gatillo de reentrenamiento y el rollback

Esto es exactamente `service/reentrenar.py`, sin modificar — el mismo script que se documenta en `PLAN_DE_ACCION.md`. Como esta copia del repositorio vive solo dentro de esta sesión de Colab, no hay ningún riesgo de tocar el modelo real del repositorio local: pueden correr esto las veces que quieran.


In [ ]:
# ── Ver el estado actual de alertas por ventana (Fase 4) ──
import pandas as pd
os.chdir(REPO_DIR)
pd.read_csv("reports/alertas_por_ventana.csv")


In [ ]:
# ── Reentrenar (ejemplo: San Juan, ventanas V2 y V3 -- ajustar segun lo que quieran demostrar) ──
os.chdir(os.path.join(REPO_DIR, "service"))
get_ipython().system('python reentrenar.py --city sj --ventanas V2 V3')


In [ ]:
# ── Ver el historial de versiones generado ──
import json
with open(os.path.join(REPO_DIR, "models", "historial_versiones.json")) as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))


In [ ]:
# ── Probar el rollback ──
os.chdir(os.path.join(REPO_DIR, "service"))
get_ipython().system('python reentrenar.py --rollback')


---
## Dashboard de monitoreo (Streamlit) — túnel público con `serveo.net`, sin cuentas ni tokens

Para ver el dashboard no hace falta crear ninguna cuenta ni pegar ningún token: usamos
`serveo.net`, un servicio de túneles gratuito que funciona con SSH (ya viene instalado en
Colab, no hay que instalar nada). Genera un link público real, distinto al proxy propio de
Colab (`google.colab.output`), que probamos primero pero no logra sostener la conexión que
Streamlit necesita para actualizar la página después de la carga inicial.

**Nota:** la primera vez que se abre el link, algún widget (por ejemplo `st.metric` o una
tabla) puede mostrar un error de "módulo importado dinámicamente" — es un problema de la
primera carga a través del túnel, no del código. Con recargar la página (Ctrl+Shift+R) o
abrirla en una ventana de incógnito se soluciona.


In [ ]:
# ── Levantar el dashboard en segundo plano ──
import subprocess, time
import requests

os.chdir(os.path.join(REPO_DIR, "dashboard"))
streamlit_proc = subprocess.Popen([
    "streamlit", "run", "dashboard.py",
    "--server.headless", "true", "--server.port", "8501",
])

def _dashboard_responde():
    try:
        return requests.get("http://127.0.0.1:8501", timeout=2)
    except requests.exceptions.ConnectionError:
        return None

# no asumimos un tiempo fijo: reintentamos hasta 20 segundos, que es lo que
# puede tardar streamlit la primera vez que se instala en esta sesion.
listo = False
for _ in range(20):
    time.sleep(1)
    if _dashboard_responde() is not None:
        listo = True
        break

if listo:
    print("Streamlit esta corriendo en el puerto 8501.")
else:
    print(
        "Aviso: streamlit todavia no respondia despues de 20 segundos. Prueben "
        "igual la celda siguiente -- si no carga, esperen unos segundos y "
        "vuelvan a correrla."
    )


In [ ]:
# ── Exponer el dashboard con un tunel publico (serveo.net, sin cuenta ni token) ──
get_ipython().system_raw('pkill -f "ssh -R 80:localhost:8501" 2>/dev/null')
time.sleep(1)
get_ipython().system_raw(
    'nohup ssh -o StrictHostKeyChecking=no -o ServerAliveInterval=30 '
    '-R 80:localhost:8501 serveo.net > /tmp/serveo.log 2>&1 < /dev/null & disown'
)

# el link tarda unos segundos en aparecer en el log -- reintentamos en vez de
# asumir un tiempo fijo
url_publica = None
for _ in range(20):
    time.sleep(1)
    try:
        with open("/tmp/serveo.log") as f:
            contenido = f.read()
    except FileNotFoundError:
        contenido = ""
    if "Forwarding HTTP traffic from" in contenido:
        url_publica = contenido.strip().splitlines()[-1].split("from")[-1].strip()
        break

if url_publica:
    print("Dashboard disponible en:", url_publica)
    print(
        "Nota: si al abrirlo algun widget muestra un error de 'modulo importado "
        "dinamicamente', recarguen la pagina (Ctrl+Shift+R) o abranla en una "
        "ventana de incognito -- es un problema de la primera carga a traves "
        "del tunel, no del codigo."
    )
else:
    print(
        "No se pudo obtener el link publico despues de 20 segundos. Revisen "
        "/tmp/serveo.log con: get_ipython().system('cat /tmp/serveo.log')"
    )


---
## Cierre y limitaciones de esta versión en Colab

- Todo lo generado en esta sesión (modelo reentrenado, base de datos de predicciones, historial de versiones) **se pierde al cerrar o reiniciar el entorno de ejecución** de Colab — es intencional (ver la nota de la Fase 5 más arriba); para conservarlo entre sesiones habría que montar Google Drive, que decidimos no hacer para mantener esto simple.
- El dashboard se ve a través de un túnel público con `serveo.net` — no depende de ninguna cuenta ni token, pero sí de que la sesión de Colab siga abierta (si se cae, hay que volver a correr las dos últimas celdas).
- Todo el código que se ejecuta aquí (`service/app.py`, `service/simular_produccion.py`, `service/reentrenar.py`, `dashboard/dashboard.py`) es exactamente el mismo que corre localmente — no se duplicó ni se reescribió nada, solo se adaptó **cómo se lanza**.


In [ ]:
# ── Limpieza: detener el dashboard y el tunel ──
streamlit_proc.terminate()
get_ipython().system_raw('pkill -f "ssh -R 80:localhost:8501" 2>/dev/null')
print("Dashboard y tunel detenidos. El servicio FastAPI del hilo de fondo se detiene solo al reiniciar el entorno de ejecucion.")
